Reproduction for 
> Hassan, W., Joolee, J.B. & Jeon, S. 
Establishing haptic texture attribute space and predicting haptic attributes from image features using 1D-CNN. 
Sci Rep 13, 11684 (2023). 
https://doi.org/10.1038/s41598-023-38929-6

In [1]:
conf = {
    'batch_size': 128,
    'epochs': 300,
    'num_workers': 0,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'dataset_output': 'four_HAs',
    'train_tag': 'reproduction',
}

In [2]:
import torch
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)

cuda


In [3]:
import pandas as pd
feature_df = pd.read_csv("data/original/FinalFeature3955_Train.csv", header=None)
X = feature_df.values
label_df = pd.read_csv("data/original/ParticipantData.csv", header=None)
Y = label_df.values
Y = Y + 50

In [4]:
from torch.utils.data import Dataset
class FullDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.FloatTensor(X)
        self.Y = torch.FloatTensor(Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

dataset = FullDataset(X, Y)

In [5]:
predictions = []
ground_truths = []
test_image_ids = []

In [6]:
# LOOCV
from tqdm import tqdm
from src.data.dataset import NormalizedSubset
from src.model.prediction.compared.cnn_1d_4ha import CNN1D4HA
from src.model.train import train_one_fold, evaluate_one_fold

for test_idx in tqdm(range(len(Y)), desc="LOOCV", unit="fold"):

    train_indices = [i for i in range(len(X)) if i != test_idx]
    test_indices = [test_idx]

    y_train_raw = Y[train_indices]   # shape: (N-1, 4)
    y_min = y_train_raw.min(axis=0)
    y_max = y_train_raw.max(axis=0)

    train_dataset = NormalizedSubset(dataset, train_indices, y_min, y_max)
    test_dataset  = NormalizedSubset(dataset, test_indices, y_min, y_max)

    model = CNN1D4HA(conf, 3955)
    model.to(device)

    model = train_one_fold(
        model=model,
        dataset=train_dataset,
        device=device,
        epochs=conf['epochs'],
        batch_size=conf['batch_size'],
        lr=conf['learning_rate'],
        weight_decay=conf['weight_decay'],
    )
        
    preds, gts = evaluate_one_fold(
        model=model,
        dataset=test_dataset,
        device=device,
        y_min=y_min,
        y_max=y_max,
    )
    
    predictions.append(preds) 
    ground_truths.append(gts)
    test_image_ids.append(test_idx)

LOOCV: 100%|██████████| 100/100 [29:03<00:00, 17.43s/fold]


In [7]:
# metrics
import numpy as np
from sklearn.metrics import mean_absolute_error
from src.utils.metrics import metrics

predictions = np.array(predictions, dtype=np.float32)
predictions = predictions.reshape(predictions.shape[0], -1)
ground_truths = np.array(ground_truths, dtype=np.float32)
ground_truths = ground_truths.reshape(ground_truths.shape[0], -1)

mae_per_output = mean_absolute_error(ground_truths, predictions, multioutput='raw_values')
rmse_per_output = np.sqrt(np.mean((ground_truths - predictions) ** 2, axis=0))

metrics(
    conf, 
    mae_per_output=mae_per_output, 
    rmse_per_output=rmse_per_output,
    predictions=predictions,
    ground_truths=ground_truths,
    test_image_ids=test_image_ids,
)


=== LOOCV Training Results ===

Results saved to experiments\runs\reproduction
  - CSV: experiments\runs\reproduction\reproduction_results.csv
  - Metrics: experiments\runs\reproduction\reproduction_metrics.json

rough-smooth MAE: 21.0153 | RMSE: 27.8790
flat-bumpy MAE: 27.6993 | RMSE: 34.9433
sticky-slippery MAE: 16.4276 | RMSE: 21.2305
hard-soft MAE: 19.7943 | RMSE: 24.4640
